In [1]:
"""
Session 27: Mask R-CNN
"""
# =============================================================================
# CELL 1 - data prep: YOLO-seg dataset -> COCO instance-segmentation JSON
# =============================================================================

import json
import math
import os
from pathlib import Path
from PIL import Image

IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".JPEG", ".PNG", ".BMP")


def polygon_area(xs, ys):
    n = len(xs)
    area = 0.0
    for i in range(n):
        j = (i + 1) % n
        area += xs[i] * ys[j] - xs[j] * ys[i]
    return abs(area) / 2.0


def discover_data_root(explicit=None):
    """Find the SkinDisNet annotation YOLO-seg dataset or use an explicit root."""
    if explicit:
        path = Path(explicit).expanduser()
        if path.is_file() and path.name.lower() == "data.yaml":
            path = path.parent
        path = path.resolve()
        if (
            (path / "data.yaml").is_file()
            and all(
                (path / split / "images").is_dir()
                and (path / split / "labels").is_dir()
                for split in ("train", "valid")
            )
        ):
            return str(path)
        raise FileNotFoundError(
            f"DATA_ROOT is not a complete YOLO-seg dataset: {path}"
        )
    candidates = []
    search_roots = [Path("/kaggle/input"), Path.cwd() / "dataset", Path.cwd()]
    for search_root in search_roots:
        if not search_root.is_dir():
            continue
        for data_yaml in search_root.rglob("data.yaml"):
            root = data_yaml.parent
            if all((root / split / "images").is_dir()
                   and (root / split / "labels").is_dir()
                   for split in ("train", "valid")):
                candidates.append(root)
    unique = sorted({path.resolve() for path in candidates if path.is_dir()})
    valid = [
        path for path in unique
        if (path / "data.yaml").is_file()
        and all(
            (path / split / "images").is_dir()
            and (path / split / "labels").is_dir()
            for split in ("train", "valid")
        )
    ]
    preferred_annotation = [
        path for path in valid
        if "".join(character.lower() for character in path.name if character.isalnum())
        .startswith("skindisnet2annotation")
    ]
    if len(preferred_annotation) == 1:
        return str(preferred_annotation[0])
    if len(preferred_annotation) > 1:
        exact = [
            path for path in preferred_annotation
            if "".join(
                character.lower() for character in path.name if character.isalnum()
            ) == "skindisnet2annotation"
        ]
        if len(exact) == 1:
            return str(exact[0])
    preferred = [path for path in unique
                 if path.name.lower() in {"skin_disease_yolo", "skin-disease-yolo"}]
    if len(preferred) == 1:
        return str(preferred[0])
    if len(valid) == 1:
        return str(valid[0])
    raise FileNotFoundError(
        "Could not uniquely locate SkinDisNet_2_Annotation. Attach it to Kaggle "
        "or set DATA_ROOT to its dataset directory. Candidates: "
        + ", ".join(str(path) for path in valid)
    )


def resolve_labels_dir(images_dir: Path) -> Path:
    parts = list(images_dir.parts)
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] == "images":
            parts[i] = "labels"
            return Path(*parts)
    return images_dir.parent / "labels"


def yolo_seg_split_to_coco(data_root, images_subdir, class_names, out_json):
    images_dir = Path(data_root) / images_subdir
    labels_dir = resolve_labels_dir(images_dir)

    if not images_dir.exists():
        raise FileNotFoundError(
            f"Images folder not found: {images_dir}\n"
            f"Check DATA_ROOT and the 'train'/'val' keys in data.yaml."
        )

    image_paths = sorted(p for p in images_dir.iterdir() if p.suffix in IMG_EXTS)

    images, annotations = [], []
    ann_id = 1
    n_missing_label = 0
    n_bbox_only_skipped = 0

    for img_id, img_path in enumerate(image_paths, start=1):
        with Image.open(img_path) as im:
            W, H = im.size

        images.append(
            {
                "id": img_id,
                "file_name": str(img_path.resolve()),
                "width": W,
                "height": H,
            }
        )

        label_path = labels_dir / (img_path.stem + ".txt")
        if not label_path.exists():
            n_missing_label += 1
            continue

        lines = [ln.strip() for ln in label_path.read_text().splitlines() if ln.strip()]
        for line in lines:
            parts = line.split()
            cls_id = int(float(parts[0]))
            coords = [float(v) for v in parts[1:]]

            if not 0 <= cls_id < len(class_names):
                raise ValueError(f"Invalid class id {cls_id} in {label_path}")
            if not all(math.isfinite(value) for value in coords):
                n_bbox_only_skipped += 1
                continue

            if len(coords) == 4:
                n_bbox_only_skipped += 1
                continue
            if len(coords) < 6 or len(coords) % 2 != 0:
                n_bbox_only_skipped += 1
                continue

            xs = [coords[i] * W for i in range(0, len(coords), 2)]
            ys = [coords[i] * H for i in range(1, len(coords), 2)]
            poly = []
            for x, y in zip(xs, ys):
                poly.extend([x, y])

            x0, x1 = min(xs), max(xs)
            y0, y1 = min(ys), max(ys)
            w, h = x1 - x0, y1 - y0
            if w < 1 or h < 1:
                continue

            annotations.append(
                {
                    "id": ann_id,
                    "image_id": img_id,
                    "category_id": cls_id,
                    "segmentation": [poly],
                    "area": polygon_area(xs, ys),
                    "bbox": [x0, y0, w, h],
                    "iscrowd": 0,
                }
            )
            ann_id += 1

    categories = [{"id": i, "name": n} for i, n in enumerate(class_names)]
    coco = {"images": images, "annotations": annotations, "categories": categories}

    out_json = Path(out_json)
    out_json.parent.mkdir(parents=True, exist_ok=True)
    out_json.write_text(json.dumps(coco))

    print(
        f"[{images_subdir}] wrote {out_json} : "
        f"{len(images)} images, {len(annotations)} instances "
        f"({n_missing_label} images had no label file -> background-only, "
        f"{n_bbox_only_skipped} lines skipped as non-polygon)"
    )
    if len(annotations) == 0:
        print(
            f"[warn] 0 instances parsed for '{images_subdir}'. This almost "
            f"certainly means the label format assumption above does not "
            f"match your files - please inspect one .txt file in "
            f"{labels_dir} by hand before trusting this run."
        )
    return str(out_json)


def convert_dataset(data_root, out_dir):
    import yaml

    data_yaml = Path(data_root) / "data.yaml"
    cfg = yaml.safe_load(data_yaml.read_text())
    raw_names = cfg.get("names")
    if isinstance(raw_names, dict):
        class_names = [str(raw_names[key]) for key in sorted(raw_names, key=lambda key: int(key))]
    elif isinstance(raw_names, (list, tuple)):
        class_names = [str(name) for name in raw_names]
    else:
        raise ValueError(f"{data_yaml} must contain a names list or mapping")
    if not class_names:
        raise ValueError(f"{data_yaml} contains no classes")
    if cfg.get("nc") is not None and int(cfg["nc"]) != len(class_names):
        raise ValueError(
            f"{data_yaml} declares nc={cfg['nc']} but has {len(class_names)} names"
        )
    print(f"[data] classes ({len(class_names)}): {class_names}")

    train_split = cfg.get("train", "train/images")
    val_split = next(
        (cfg[key] for key in ("val", "valid", "validation") if cfg.get(key)),
        None,
    )
    if not val_split:
        raise ValueError(
            f"{data_yaml} must define a validation split as 'val', 'valid', "
            "or 'validation'."
        )
    train_json = yolo_seg_split_to_coco(
        data_root, train_split, class_names, Path(out_dir) / "train.json"
    )
    val_json = yolo_seg_split_to_coco(
        data_root, val_split, class_names, Path(out_dir) / "val.json"
    )
    return train_json, val_json


print("[cell 1] data-prep functions defined.")


# =============================================================================
# CELL 2 - Mask R-CNN (torchvision maskrcnn_resnet50_fpn_v2) training code
# =============================================================================
# This only DEFINES functions/classes. Nothing runs yet - CELL 3 calls
# train_fold(...) explicitly. (This is what fixes the earlier crash: the
# old version's "if __name__ == '__main__': main()" ran automatically the
# moment the cell executed, and main() tried to read command-line arguments
# that don't exist in a notebook kernel.)

import copy
import random
import time

import numpy as np
import torch
import torchvision
from PIL import Image
from pycocotools import mask as mask_util
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor


class CocoSegDataset(Dataset):
    """Reads a COCO instance-seg JSON. Labels are shifted +1 for torchvision
    (0 is reserved for background); category_id in the JSON stays 0-indexed
    for scoring, so predictions are shifted back -1 before COCOeval."""

    def __init__(self, json_path, train=False):
        self.json_path = str(json_path)
        self.coco = COCO(str(json_path))
        self.ids = sorted(self.coco.imgs.keys())
        self.train = train
        self.cats = sorted(self.coco.cats.keys())

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        info = self.coco.imgs[img_id]
        img = Image.open(info["file_name"]).convert("RGB")
        W, H = img.size

        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)

        boxes, labels, masks, areas, iscrowd = [], [], [], [], []
        for a in anns:
            x, y, w, h = a["bbox"]
            if w < 1 or h < 1:
                continue
            boxes.append([x, y, x + w, y + h])
            labels.append(a["category_id"] + 1)  # +1: background is 0
            rle = mask_util.frPyObjects(a["segmentation"], H, W) if not isinstance(
                a["segmentation"], dict
            ) else a["segmentation"]
            m = mask_util.decode(rle if isinstance(rle, dict) else mask_util.merge(rle))
            masks.append(m)
            areas.append(a["area"])
            iscrowd.append(a.get("iscrowd", 0))

        img_t = torchvision.transforms.functional.to_tensor(img)

        if self.train and random.random() < 0.5 and boxes:
            img_t = torch.flip(img_t, dims=[2])
            boxes = [[W - x2, y1, W - x1, y2] for x1, y1, x2, y2 in boxes]
            masks = [np.fliplr(m).copy() for m in masks]

        if boxes:
            target = {
                "boxes": torch.as_tensor(boxes, dtype=torch.float32),
                "labels": torch.as_tensor(labels, dtype=torch.int64),
                "masks": torch.as_tensor(np.stack(masks), dtype=torch.uint8),
                "image_id": torch.tensor([img_id]),
                "area": torch.as_tensor(areas, dtype=torch.float32),
                "iscrowd": torch.as_tensor(iscrowd, dtype=torch.int64),
            }
        else:
            target = {
                "boxes": torch.zeros((0, 4), dtype=torch.float32),
                "labels": torch.zeros((0,), dtype=torch.int64),
                "masks": torch.zeros((0, H, W), dtype=torch.uint8),
                "image_id": torch.tensor([img_id]),
                "area": torch.zeros((0,), dtype=torch.float32),
                "iscrowd": torch.zeros((0,), dtype=torch.int64),
            }
        return img_t, target


def collate_fn(batch):
    return tuple(zip(*batch))


def build_model(num_classes, imgsz):
    weights = torchvision.models.detection.MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    model = torchvision.models.detection.maskrcnn_resnet50_fpn_v2(
        weights=weights,
        min_size=int(imgsz * 0.875),
        max_size=int(imgsz * 1.67),
        box_detections_per_img=50,
    )
    in_feat = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_classes)
    in_feat_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_feat_mask, 256, num_classes)
    return model


@torch.no_grad()
def run_inference(model, loader, device):
    model.eval()
    results = []
    for images, targets in loader:
        images = [im.to(device) for im in images]
        outputs = model(images)
        for tgt, out in zip(targets, outputs):
            img_id = int(tgt["image_id"].item())
            boxes = out["boxes"].cpu().numpy()
            scores = out["scores"].cpu().numpy()
            labels = out["labels"].cpu().numpy()
            masks = out["masks"].cpu().numpy()
            for b, s, l, m in zip(boxes, scores, labels, masks):
                if s < 0.05:
                    continue
                binm = (m[0] > 0.5).astype(np.uint8)
                rle = mask_util.encode(np.asfortranarray(binm))
                rle["counts"] = rle["counts"].decode("ascii")
                x1, y1, x2, y2 = b.tolist()
                results.append(
                    {
                        "image_id": img_id,
                        "category_id": int(l) - 1,
                        "bbox": [x1, y1, x2 - x1, y2 - y1],
                        "score": float(s),
                        "segmentation": rle,
                    }
                )
    return results


def decode_ann_mask(seg, H, W):
    if isinstance(seg, dict):
        return mask_util.decode(seg).astype(bool)
    rle = mask_util.frPyObjects(seg, H, W)
    return mask_util.decode(mask_util.merge(rle)).astype(bool)


def strip_segmentation(r):
    return {k: v for k, v in r.items() if k != "segmentation"}


def coco_eval(coco_gt, results, iou_type):
    if not results:
        return None
    coco_dt = coco_gt.loadRes(copy.deepcopy(results))
    ev = COCOeval(coco_gt, coco_dt, iouType=iou_type)
    ev.evaluate()
    ev.accumulate()
    ev.summarize()
    return ev


def per_class_ap(ev, cat_ids, names):
    if ev is None:
        return {n: None for n in names}
    out = {}
    for i, cid in enumerate(cat_ids):
        s = ev.eval["precision"][:, :, i, 0, -1]
        s = s[s > -1]
        out[names[i]] = float(s.mean()) if s.size else 0.0
    return out


def dice_iou_per_class(coco_gt, results, cat_ids, names, score_thresh=0.5, iou_match=0.5):
    by_img = {}
    for r in results:
        if r["score"] < score_thresh:
            continue
        by_img.setdefault(r["image_id"], []).append(r)

    dice_sums = {c: [] for c in cat_ids}
    for img_id, preds in by_img.items():
        ann_ids = coco_gt.getAnnIds(imgIds=img_id)
        gts = coco_gt.loadAnns(ann_ids)
        info = coco_gt.imgs[img_id]
        H, W = info["height"], info["width"]
        gt_masks = [decode_ann_mask(g["segmentation"], H, W) for g in gts]
        used = set()
        for p in sorted(preds, key=lambda r: -r["score"]):
            pm = decode_ann_mask(p["segmentation"], H, W)
            best_iou, best_j = 0.0, -1
            for j, (g, gm) in enumerate(zip(gts, gt_masks)):
                if j in used or g["category_id"] != p["category_id"]:
                    continue
                inter = np.logical_and(pm, gm).sum()
                union = np.logical_or(pm, gm).sum()
                iou = inter / union if union else 0.0
                if iou > best_iou:
                    best_iou, best_j = iou, j
            if best_iou >= iou_match:
                used.add(best_j)
                gm = gt_masks[best_j]
                inter = np.logical_and(pm, gm).sum()
                dice = 2 * inter / (pm.sum() + gm.sum()) if (pm.sum() + gm.sum()) else 0.0
                dice_sums[p["category_id"]].append((dice, best_iou))

    out = {}
    for c, n in zip(cat_ids, names):
        vals = dice_sums[c]
        out[n] = {
            "dice": float(np.mean([v[0] for v in vals])) if vals else 0.0,
            "iou": float(np.mean([v[1] for v in vals])) if vals else 0.0,
            "n_matched": len(vals),
        }
    return out


def make_plots(history, class_metrics, out_dir, ds_val, model, device):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    epochs = [h["epoch"] for h in history]
    plt.figure(figsize=(6, 4))
    plt.plot(epochs, [h["train_loss"] for h in history], label="train loss")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("Training loss")
    plt.legend(); plt.tight_layout()
    plt.savefig(out_dir / "loss_curve.png", dpi=150); plt.close()

    names = list(class_metrics.keys())
    dice = [class_metrics[n]["dice"] for n in names]
    iou = [class_metrics[n]["iou"] for n in names]
    x = np.arange(len(names))
    plt.figure(figsize=(7, 4))
    plt.bar(x - 0.2, dice, width=0.4, label="Dice")
    plt.bar(x + 0.2, iou, width=0.4, label="IoU")
    plt.xticks(x, names, rotation=20, ha="right")
    plt.ylabel("score"); plt.title("Per-class Dice / IoU (matched @ IoU>0.5)")
    plt.legend(); plt.tight_layout()
    plt.savefig(out_dir / "per_class_ap.png", dpi=150); plt.close()

    model.eval()
    n_show = min(4, len(ds_val))
    fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 4))
    axes = [axes] if n_show == 1 else axes
    cmap = plt.get_cmap("tab10")
    with torch.no_grad():
        for i in range(n_show):
            img, _ = ds_val[i]
            out = model([img.to(device)])[0]
            im = img.permute(1, 2, 0).cpu().numpy()
            axes[i].imshow(im)
            for m, s, l in zip(out["masks"].cpu().numpy(),
                               out["scores"].cpu().numpy(),
                               out["labels"].cpu().numpy()):
                if s < 0.5:
                    continue
                axes[i].contour(m[0] > 0.5, colors=[cmap((l - 1) % 10)], linewidths=1.5)
            axes[i].axis("off")
    plt.tight_layout()
    plt.savefig(out_dir / "sample_predictions.png", dpi=150); plt.close()


def _make_scaler(amp_device, enabled):
    try:
        return torch.amp.GradScaler(amp_device, enabled=enabled)
    except TypeError:
        return torch.cuda.amp.GradScaler(enabled=enabled)


def _autocast_ctx(amp_device, enabled):
    try:
        return torch.amp.autocast(amp_device, enabled=enabled)
    except TypeError:
        return torch.cuda.amp.autocast(enabled=enabled)


def train_fold(train_json, val_json, output, imgsz=800, batch=4, epochs=24, lr=0.005,
               workers=2, budget_min=150.0, seed=0, resume=False, eval_only=False):
    """Call this directly from a notebook cell - no argparse involved, so
    there is nothing here that can misread the kernel's own launch line."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out_dir = Path(output); out_dir.mkdir(parents=True, exist_ok=True)
    print(f"[env] device={device} "
          f"({torch.cuda.get_device_name(0) if device.type=='cuda' else 'cpu'})")

    ds_train = CocoSegDataset(train_json, train=True)
    ds_val = CocoSegDataset(val_json, train=False)
    cat_ids = ds_train.cats
    names = [ds_train.coco.cats[c]["name"] for c in cat_ids]
    num_classes = len(cat_ids) + 1
    print(f"[data] {len(ds_train)} train / {len(ds_val)} val | classes: {names}")

    dl_train = DataLoader(ds_train, batch_size=batch, shuffle=True,
                           num_workers=workers, collate_fn=collate_fn,
                           drop_last=True, persistent_workers=workers > 0)
    dl_val = DataLoader(ds_val, batch_size=1, shuffle=False,
                         num_workers=workers, collate_fn=collate_fn,
                         persistent_workers=workers > 0)

    model = build_model(num_classes, imgsz).to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=[int(epochs * 0.7), int(epochs * 0.9)], gamma=0.1
    )
    amp_device = "cuda" if device.type == "cuda" else "cpu"
    scaler = _make_scaler(amp_device, enabled=device.type == "cuda")

    start_epoch, history = 0, []
    last_ckpt = out_dir / "last.pth"
    if resume and last_ckpt.exists():
        ck = torch.load(last_ckpt, map_location=device)
        model.load_state_dict(ck["model"])
        optimizer.load_state_dict(ck["optim"])
        # Deliberately NOT scheduler.load_state_dict(ck["sched"]) here: that
        # would restore the milestones computed under whatever `epochs` value
        # was used when the checkpoint was saved. If you resume with a
        # different `epochs` (e.g. you originally trained with epochs=24 and
        # are now continuing with epochs=40), the scheduler above was already
        # built fresh with the NEW milestones - stepping it forward the same
        # number of times the old run did keeps that new schedule intact
        # instead of silently reverting to the old one.
        old_epoch = ck["epoch"]
        for _ in range(old_epoch + 1):
            scheduler.step()
        scaler.load_state_dict(ck["scaler"])
        start_epoch = old_epoch + 1
        history = ck.get("history", [])
        print(f"[resume] continuing from epoch {start_epoch} "
              f"(LR schedule recomputed for epochs={epochs}, "
              f"current lr={scheduler.get_last_lr()[0]:.5f})")

    if eval_only:
        results = run_inference(model, dl_val, device)
        return _finalize(ds_val.coco, results, cat_ids, names, out_dir, history, model, ds_val, device)

    t0 = time.time()
    deadline = t0 + budget_min * 60
    stopped_early = False

    for epoch in range(start_epoch, epochs):
        model.train()
        running = 0.0
        n_batches = 0
        for images, targets in dl_train:
            if time.time() > deadline:
                stopped_early = True
                break
            images = [im.to(device) for im in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            optimizer.zero_grad(set_to_none=True)
            with _autocast_ctx(amp_device, enabled=device.type == "cuda"):
                loss_dict = model(images, targets)
                loss = sum(loss_dict.values())
            if not torch.isfinite(loss):
                print(f"[warn] non-finite loss at epoch {epoch}, skipping batch")
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, 5.0)
            scaler.step(optimizer)
            scaler.update()
            running += loss.item()
            n_batches += 1

        scheduler.step()
        train_loss = running / max(n_batches, 1)
        history.append({"epoch": epoch, "train_loss": train_loss,
                         "lr": scheduler.get_last_lr()[0],
                         "elapsed_min": (time.time() - t0) / 60})
        print(f"[epoch {epoch}] loss={train_loss:.4f} "
              f"lr={scheduler.get_last_lr()[0]:.5f} "
              f"elapsed={(time.time()-t0)/60:.1f}min")

        torch.save({"model": model.state_dict(), "optim": optimizer.state_dict(),
                    "sched": scheduler.state_dict(), "scaler": scaler.state_dict(),
                    "epoch": epoch, "history": history}, last_ckpt)

        if stopped_early:
            print(f"[budget] stopped at epoch {epoch} "
                  f"({(time.time()-t0)/60:.1f} min used). Re-run with resume=True.")
            break

    results = run_inference(model, dl_val, device)
    return _finalize(ds_val.coco, results, cat_ids, names, out_dir, history, model, ds_val, device)


def _print_history_table(history):
    """Prints the epoch-by-epoch training history as a table. Uses pandas
    if it's available (it is on virtually every Kaggle image) so it renders
    as a proper table in the notebook output; falls back to a manually
    aligned plain-text table if pandas is missing."""
    if not history:
        print("[history] no epochs were completed yet.")
        return
    try:
        import pandas as pd
        df = pd.DataFrame(history)[["epoch", "train_loss", "lr", "elapsed_min"]].round(4)
        print("\nTraining history")
        try:
            from IPython.display import display
            display(df)
        except ImportError:
            print(df.to_string(index=False))
        return df
    except ImportError:
        header = f"{'epoch':>5}  {'train_loss':>10}  {'lr':>8}  {'elapsed_min':>11}"
        print("\nTraining history")
        print(header)
        print("-" * len(header))
        for h in history:
            print(f"{h['epoch']:>5}  {h['train_loss']:>10.4f}  {h['lr']:>8.5f}  {h['elapsed_min']:>11.1f}")
        return None


def _print_results_tables(metrics):
    """Prints the final overall metrics and per-class metrics as two
    separate tables instead of a raw JSON dump. Same pandas-with-fallback
    approach as _print_history_table."""
    overall_keys = ["bbox_AP", "bbox_AP50", "segm_AP", "segm_AP50", "n_val_images", "n_predictions"]
    per_class_ap = metrics.get("segm_AP_per_class", {}) or {}
    dice_iou = metrics.get("dice_iou_per_class", {}) or {}
    class_names = list(per_class_ap.keys())

    try:
        import pandas as pd
        try:
            from IPython.display import display
        except ImportError:
            display = None

        overall_df = pd.DataFrame(
            [{"metric": k, "value": metrics.get(k)} for k in overall_keys]
        )
        print("\nOverall metrics")
        (display or print)(overall_df)

        rows = []
        for name in class_names:
            di = dice_iou.get(name, {})
            rows.append(
                {
                    "class": name,
                    "segm_AP": per_class_ap.get(name),
                    "dice": di.get("dice"),
                    "iou": di.get("iou"),
                    "n_matched": di.get("n_matched"),
                }
            )
        per_class_df = pd.DataFrame(rows).round(4)
        print("\nPer-class metrics")
        (display or print)(per_class_df)
        return overall_df, per_class_df
    except ImportError:
        print("\nOverall metrics")
        for k in overall_keys:
            print(f"  {k:<14}: {metrics.get(k)}")

        print("\nPer-class metrics")
        header = f"{'class':<16}{'segm_AP':>10}{'dice':>10}{'iou':>10}{'n_matched':>11}"
        print(header)
        print("-" * len(header))
        for name in class_names:
            di = dice_iou.get(name, {})
            print(
                f"{name:<16}{(per_class_ap.get(name) or 0):>10.4f}"
                f"{(di.get('dice') or 0):>10.4f}{(di.get('iou') or 0):>10.4f}"
                f"{(di.get('n_matched') or 0):>11}"
            )
        return None, None


def _finalize(coco_gt, results, cat_ids, names, out_dir, history, model, ds_val, device):
    (out_dir / "predictions_val.json").write_text(json.dumps(results))

    dice_iou = dice_iou_per_class(coco_gt, results, cat_ids, names) if results else {
        n: {"dice": 0.0, "iou": 0.0, "n_matched": 0} for n in names
    }

    bbox_results = [strip_segmentation(r) for r in results]
    bbox_ev = coco_eval(coco_gt, bbox_results, "bbox") if results else None
    segm_ev = coco_eval(coco_gt, results, "segm") if results else None

    metrics = {
        "bbox_AP": float(bbox_ev.stats[0]) if bbox_ev else None,
        "bbox_AP50": float(bbox_ev.stats[1]) if bbox_ev else None,
        "segm_AP": float(segm_ev.stats[0]) if segm_ev else None,
        "segm_AP50": float(segm_ev.stats[1]) if segm_ev else None,
        "segm_AP_per_class": per_class_ap(segm_ev, cat_ids, names),
        "n_val_images": len(coco_gt.imgs),
        "n_predictions": len(results),
    }
    metrics["dice_iou_per_class"] = dice_iou

    (out_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
    (out_dir / "history.json").write_text(json.dumps(history, indent=2))

    history_df = _print_history_table(history)
    overall_df, per_class_df = _print_results_tables(metrics)

    try:
        import pandas as pd
        if history_df is None:
            history_df = pd.DataFrame(history)
        if overall_df is None:
            overall_df = pd.DataFrame(
                [{"metric": k, "value": metrics.get(k)}
                 for k in ["bbox_AP", "bbox_AP50", "segm_AP", "segm_AP50", "n_val_images", "n_predictions"]]
            )
        if per_class_df is None:
            rows = []
            for name in metrics.get("segm_AP_per_class", {}) or {}:
                di = (metrics.get("dice_iou_per_class", {}) or {}).get(name, {})
                rows.append({"class": name, "segm_AP": metrics["segm_AP_per_class"][name],
                             "dice": di.get("dice"), "iou": di.get("iou"), "n_matched": di.get("n_matched")})
            per_class_df = pd.DataFrame(rows)
        history_df.to_csv(out_dir / "history_table.csv", index=False)
        overall_df.to_csv(out_dir / "overall_metrics_table.csv", index=False)
        per_class_df.to_csv(out_dir / "per_class_metrics_table.csv", index=False)
        print(f"\n[ok] table CSVs written to {out_dir}")
    except ImportError:
        pass

    try:
        ds_val_fresh = CocoSegDataset(ds_val.json_path, train=False)
        make_plots(history, dice_iou, out_dir, ds_val_fresh, model, device)
        print(f"[ok] plots written to {out_dir}")
    except Exception as e:
        print(f"[warn] plotting failed (non-fatal): {e}")

    return metrics


print("[cell 2] training functions defined.")


# =============================================================================
# CELL 3 - actually run the pipeline: convert data, then train
# =============================================================================
# EDIT DATA_ROOT (and, if you're on Colab rather than Kaggle, COCO_DIR /
# RUN_DIR too - see the platform note at the top of this file).

DATA_ROOT = discover_data_root(os.environ.get("DATA_ROOT") or None)
SESSION_WORK_DIR = "/kaggle/working/session27_skin_disnet_6000"
COCO_DIR = f"{SESSION_WORK_DIR}/coco"
RUN_DIR = f"{SESSION_WORK_DIR}/runs/fold0"

EPOCHS = 40
BATCH_SIZE = 4
IMG_SIZE = 800
# Kaggle GPU notebook sessions have a 12-hour (720 min) hard runtime limit
# per session, on top of a separate ~30 hours/week total GPU quota - so a
# single run can't exceed ~720 min regardless of this setting. Your last run
# did ~18.75 min/epoch, so 660 min leaves about an hour of buffer for setup,
# data loading, and the final eval/plotting step (which itself took a few
# minutes in your last run), while still fitting inside one 12-hour session.
# This 18.75 min/epoch rate is measured from one run on your data, not a
# universal constant - if your actual per-epoch time differs, adjust this
# number, or just let it stop early and resume again (RESUME=True below
# handles that automatically).
BUDGET_MIN = 660.0
RESUME = True  # continue from /kaggle/working/runs/fold0/last.pth instead of retraining from scratch

try:
    import pycocotools  # noqa: F401
except ImportError:
    print("[setup] pycocotools not found, installing...")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocotools"], check=True)

train_json, val_json = convert_dataset(data_root=DATA_ROOT, out_dir=COCO_DIR)

metrics = train_fold(
    train_json=train_json,
    val_json=val_json,
    output=RUN_DIR,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    epochs=EPOCHS,
    budget_min=BUDGET_MIN,
    resume=RESUME,
)
print("[done] final metrics:")
print(metrics)


[cell 1] data-prep functions defined.
[cell 2] training functions defined.
[data] classes (6): ['Atopic Dermatitis', 'Contact Dermatitis', 'Eczema', 'Scabies', 'Seborrheic Dermatitis', 'Tinea Corporis']
[train/images] wrote /kaggle/working/session27_skin_disnet_6000/coco/train.json : 4000 images, 5758 instances (0 images had no label file -> background-only, 0 lines skipped as non-polygon)
[valid/images] wrote /kaggle/working/session27_skin_disnet_6000/coco/val.json : 1000 images, 1351 instances (0 images had no label file -> background-only, 0 lines skipped as non-polygon)
[env] device=cuda (Tesla T4)
loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
[data] 4000 train / 1000 val | classes: ['Atopic Dermatitis', 'Contact Dermatitis', 'Eczema', 'Scabies', 'Seborrheic Dermatitis', 'Tinea Corporis']
Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_v2_c

100%|██████████| 177M/177M [00:00<00:00, 250MB/s]


[epoch 0] loss=0.2931 lr=0.00500 elapsed=8.2min
[epoch 1] loss=0.1922 lr=0.00500 elapsed=16.4min
[epoch 2] loss=0.1730 lr=0.00500 elapsed=24.6min
[epoch 3] loss=0.1604 lr=0.00500 elapsed=32.8min
[epoch 4] loss=0.1490 lr=0.00500 elapsed=41.0min
[epoch 5] loss=0.1403 lr=0.00500 elapsed=49.2min
[epoch 6] loss=0.1316 lr=0.00500 elapsed=57.4min
[epoch 7] loss=0.1215 lr=0.00500 elapsed=65.6min
[epoch 8] loss=0.1167 lr=0.00500 elapsed=73.8min
[epoch 9] loss=0.1071 lr=0.00500 elapsed=82.0min
[epoch 10] loss=0.1010 lr=0.00500 elapsed=90.2min
[epoch 11] loss=0.0961 lr=0.00500 elapsed=98.4min
[epoch 12] loss=0.0904 lr=0.00500 elapsed=106.6min
[epoch 13] loss=0.0861 lr=0.00500 elapsed=114.8min
[epoch 14] loss=0.0823 lr=0.00500 elapsed=123.0min
[epoch 15] loss=0.0815 lr=0.00500 elapsed=131.2min
[epoch 16] loss=0.0758 lr=0.00500 elapsed=139.4min
[epoch 17] loss=0.0739 lr=0.00500 elapsed=147.6min
[epoch 18] loss=0.0707 lr=0.00500 elapsed=155.8min
[epoch 19] loss=0.0696 lr=0.00500 elapsed=164.0min
[ep

,epoch,train_loss,lr,elapsed_min
0,0,0.2931,0.0050,8.1663
1,1,0.1922,0.0050,16.3519
2,2,0.1730,0.0050,24.5522
3,3,0.1604,0.0050,32.7553
4,4,0.1490,0.0050,40.9707
5,5,0.1403,0.0050,49.1770
6,6,0.1316,0.0050,57.3961
7,7,0.1215,0.0050,65.6147
8,8,0.1167,0.0050,73.8224
9,9,0.1071,0.0050,82.0321



Overall metrics


,metric,value
0,bbox_AP,0.184286
1,bbox_AP50,0.189321
2,segm_AP,0.181070
3,segm_AP50,0.188559
4,n_val_images,1000.000000
5,n_predictions,3388.000000



Per-class metrics


,class,segm_AP,dice,iou,n_matched
0,Atopic Dermatitis,0.0000,0.0000,0.0000,0
1,Contact Dermatitis,0.3254,0.9842,0.9706,279
2,Eczema,0.1967,0.9752,0.9543,130
3,Scabies,0.3630,0.9847,0.9706,117
4,Seborrheic Dermatitis,0.0539,0.9880,0.9764,6
5,Tinea Corporis,0.1474,0.9845,0.9707,62



[ok] table CSVs written to /kaggle/working/session27_skin_disnet_6000/runs/fold0
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
[ok] plots written to /kaggle/working/session27_skin_disnet_6000/runs/fold0
[done] final metrics:
{'bbox_AP': 0.1842862888319926, 'bbox_AP50': 0.1893212684433317, 'segm_AP': 0.1810696465503269, 'segm_AP50': 0.18855938929433189, 'segm_AP_per_class': {'Atopic Dermatitis': 0.0, 'Contact Dermatitis': 0.3254092108057584, 'Eczema': 0.19669189363584721, 'Scabies': 0.36302705560973325, 'Seborrheic Dermatitis': 0.05392005867253392, 'Tinea Corporis': 0.14736966057808867}, 'n_val_images': 1000, 'n_predictions': 3388, 'dice_iou_per_class': {'Atopic Dermatitis': {'dice': 0.0, 'iou': 0.0, 'n_matched': 0}, 'Contact Dermatitis': {'dice': 0.9842072118795628, 'iou': 0.9705540793783405, 'n_matched': 279}, 'Eczema': {'dice': 0.9751820661162857, 'iou': 0.9542570291796705, 'n_matched': 130}, 'Scabies': {'dice': 0.9846620824289632, 'iou': 0.97063